# Procesos de Decisión de Markov con dos estados

In [29]:
import numpy as np

# =========================
# MDP del ejemplo
# =========================
S1, S2 = 0, 1                 # estados
A11, A12, A21 = 0, 1, 2       # acciones

state_names = {S1: "s1", S2: "s2"}
action_names = {A11: "a11", A12: "a12", A21: "a21"}

# Transiciones p(s'|s,a)
P = {
    (S1, A11): [(S1, 0.5), (S2, 0.5)],
    (S1, A12): [(S2, 1.0)],
    (S2, A21): [(S2, 1.0)],
}
# Recompensa r(s,a)
# -------------------------
def reward(s, a):
    if s == S1 and a == A11:
        return 5.0
    if s == S1 and a == A12:
        return 10.0
    if s == S2 and a == A21:
        return -1.0
    raise ValueError("Par (s,a) inválido")


In [30]:
#transiciones de estado
def next_state(rng, s, a):
    if s == S1 and a == A11:
        # 50% s1, 50% s2
        return rng.choice([S1, S2], p=[0.5, 0.5])
    
    if s == S1 and a == A12:
        # siempre a s2
        return S2
    
    if s == S2 and a == A21:
        # siempre permanece en s2
        return S2

In [31]:
def policy_MD(t, s):
    """Política determinista Markov: d1, d2."""
    if t == 1:
        return A11 if s == S1 else A21
    elif t == 2:
        return A12 if s == S1 else A21
    else:
        raise ValueError("t debe ser 1 o 2 (N=3).")


In [7]:
def policy_MR(rng, t, s):
    """Política aleatorizada Markov: q1, q2."""
    if s == S2:
        return A21

    # s == S1
    if t == 1:
        return rng.choice([A11, A12], p=[0.7, 0.3])
    elif t == 2:
        return rng.choice([A11, A12], p=[0.4, 0.6])
    else:
        raise ValueError("t debe ser 1 o 2 (N=3).")


In [32]:
# =========================
# Simulación de UNA trayectoria
# =========================
def simulate_episode(policy_type, s0):
    
    s = s0
    total_reward = 0
    rng = np.random.default_rng()
    
    for t in [1, 2]:
        
        if policy_type == "MD":
            a = policy_MD(t, s)
        else:
            a = policy_MR(rng, t, s)
        
        total_reward += reward(s, a)
        s = next_state(rng, s, a)
    
    return total_reward


# =========================
# Monte Carlo 
# =========================
def monte_carlo(n_runs=10000, s0=S1, policy_type="MD"):
    
    totals = []
    
    for _ in range(n_runs):
        total = simulate_episode(policy_type, s0)  # ← SOLO 2 argumentos
        totals.append(total)
    
    totals = np.array(totals)
    
    return {
        "mean": totals.mean(),
        "std": totals.std(),
        "min": totals.min(),
        "max": totals.max()
    }


In [33]:
# =========================
# Ejemplo: una trayectoria
print("Una trayectoria con política MD:")
print("Recompensa total:", simulate_episode("MD", S1))

print("\nUna trayectoria con política MR:")
print("Recompensa total:", simulate_episode("MR", S1))


# =========================
# Comparación Monte Carlo
# =========================
results_MD = monte_carlo(20000, S1, "MD")
results_MR = monte_carlo(20000, S1, "MR")

print("\n=== Resultados Monte Carlo (inicio en s1) ===")
print(f"MD -> media: {results_MD['mean']:.4f}, std: {results_MD['std']:.4f}")
print(f"MR -> media: {results_MR['mean']:.4f}, std: {results_MR['std']:.4f}")



Una trayectoria con política MD:
Recompensa total: 4.0

Una trayectoria con política MR:
Recompensa total: 4.0

=== Resultados Monte Carlo (inicio en s1) ===
MD -> media: 9.6111, std: 5.4989
MR -> media: 8.6867, std: 4.0507


# Inventario con un solo producto

In [39]:
import numpy as np

# =========================
# Demanda: ejemplo discreto
# =========================
# D_t ∈ {0,1,2} con prob 1/3 
DEMAND_VALUES = np.array([0, 1, 2])
DEMAND_PROBS  = np.array([1/3, 1/3, 1/3])

def sample_demand(rng):
    return rng.choice(DEMAND_VALUES, p=DEMAND_PROBS)

# =========================
# Costos (ejemplo)
# =========================
K = 8.0   # costo fijo si ordenas (a_t > 0)
c = 1.0   # costo unitario por ordenar
h = 1.0   # holding sobre inventario final X_{t+1}
p = 5.0   # penalización por faltante (lost sales): (D - (X+a))^+

# =========================
# Política umbral (s_t, S_t)
# decisiones en t=1,2
# =========================
def order_threshold(x, t, s1=1, S1=3, s2=0, S2=2):
    if t == 1:
        return max(0, S1 - x) if x <= s1 else 0
    if t == 2:
        return max(0, S2 - x) if x <= s2 else 0
    return 0  # t=3: no ordenamos


In [37]:
# =========================
# Simular una trayectoria (3 periodos)
# =========================
def simulate_one_run(rng, x1=0, s1=1, S1=3, s2=0, S2=2, verbose=False):
    x = x1
    total_cost = 0.0

    for t in [1, 2, 3]:
        # acción (orden) según política; en t=3 es 0
        a = order_threshold(x, t, s1=s1, S1=S1, s2=s2, S2=S2)

        # demanda
        d = sample_demand(rng)

        # inventario disponible después de ordenar
        y = x + a

        # ventas y faltante (lost sales)
        shortage = max(0, d - y)

        # inventario final (lo que queda)
        x_next = max(0, y - d)

        # costo del periodo
        setup = K if a > 0 else 0.0
        period_cost = setup + c * a + h * x_next + p * shortage

        total_cost += period_cost

        if verbose:
            print(f"t={t}, x={x}, a={a}, d={d}, x_next={x_next}, shortage={shortage}, cost={period_cost:.2f}")

        x = x_next  # avanzar al siguiente periodo

    return total_cost

In [38]:
# =========================
# Monte Carlo
# =========================
def monte_carlo(n_runs=10000, x1=0, s1=1, S1=3, s2=0, S2=2, seed=123):
    rng = np.random.default_rng(seed)
    costs = np.array([simulate_one_run(rng, x1, s1, S1, s2, S2) for _ in range(n_runs)], dtype=float)
    return {
        "mean": costs.mean(),
        "std": costs.std(ddof=1),
        "min": costs.min(),
        "max": costs.max()
    }

# =========================
# Prueba rápida
# =========================
rng = np.random.default_rng(7)
print("Una trayectoria (verbose) con política (s1,S1)=(1,3), (s2,S2)=(0,2):")
_ = simulate_one_run(rng, x1=0, s1=1, S1=3, s2=0, S2=2, verbose=True)

res = monte_carlo(n_runs=20000, x1=0, s1=1, S1=3, s2=0, S2=2, seed=2026)
print("\n=== Monte Carlo (costo total en 3 periodos) ===")
print(f"media={res['mean']:.4f}, std={res['std']:.4f}, min={res['min']:.2f}, max={res['max']:.2f}")

Una trayectoria (verbose) con política (s1,S1)=(1,3), (s2,S2)=(0,2):
t=1, x=0, a=3, d=1, x_next=2, shortage=0, cost=13.00
t=2, x=2, a=0, d=2, x_next=0, shortage=0, cost=0.00
t=3, x=0, a=0, d=2, x_next=0, shortage=2, cost=10.00

=== Monte Carlo (costo total en 3 periodos) ===
media=17.4295, std=3.3279, min=12.00, max=27.00
